# 00 — Download e separação dos dados

**Objetivo:** baixar a base *Bank Account Fraud (BAF) — NeurIPS 2022* e separá-la, **já na origem**, em treino (meses 0–5) e teste (meses 6–7).

**Por que separar aqui, e não depois?** Se o teste já nasce num arquivo separado, os notebooks seguintes carregam só o treino e não têm como olhar o teste por acidente. É a forma mais simples de garantir que nenhuma decisão de EDA ou modelagem seja influenciada por ele.

**Saídas** (na pasta `data/`, que não vai para o Git):

| Arquivo | Conteúdo |
|---|---|
| `Base.csv` | arquivo original |
| `train.parquet` | meses 0–5 (treino) |
| `test.parquet` | meses 6–7 (teste — só usado na avaliação final) |

> **Por que Parquet?** Lê muito mais rápido que CSV, ocupa menos espaço e preserva os tipos das colunas.

In [1]:
import os
import sys
import shutil
from pathlib import Path

import pandas as pd

# --- Caminhos portáveis: funciona localmente (VS Code) e no Google Colab ---
EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/fraude-abertura-conta")
else:
    # Localmente o notebook roda de dentro de notebooks/; a raiz do repositório é a pasta acima
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAIZ = Path(os.getenv("FRAUDE_RAIZ", RAIZ))  # opcional: sobrescrever por variável de ambiente
DATA_DIR = RAIZ / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ambiente: {'Colab' if EM_COLAB else 'local'} | dados em: {DATA_DIR}")

Ambiente: local | dados em: c:\Users\joaog\Documents\Projetos\fraude-abertura-conta\data


## 1. Download

O `kagglehub` baixa datasets públicos do Kaggle sem precisar de login. O pacote vem com 6 variantes da base (`Base` + `Variant I` a `V`, que têm vieses injetados de propósito para pesquisa em justiça algorítmica); usamos apenas a **`Base.csv`**.

Se o arquivo já existir em `data/`, o download é pulado.

In [2]:
DATASET = "sgpjesus/bank-account-fraud-dataset-neurips-2022"
ARQUIVO_BRUTO = DATA_DIR / "Base.csv"

if ARQUIVO_BRUTO.exists():
    print(f"{ARQUIVO_BRUTO.name} já existe — download pulado.")
else:
    import kagglehub
    pasta_cache = Path(kagglehub.dataset_download(DATASET))
    shutil.copy(pasta_cache / "Base.csv", ARQUIVO_BRUTO)
    print(f"Baixado e copiado para {ARQUIVO_BRUTO}")

c:\Users\joaog\Documents\Projetos\fraude-abertura-conta\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 532M/532M [00:18<00:00, 30.6MB/s] 

Extracting files...


Baixado e copiado para c:\Users\joaog\Documents\Projetos\fraude-abertura-conta\data\Base.csv


## 2. Checagem de integridade

Antes de usar o arquivo, confirmamos que ele é o que esperamos. Os `assert` interrompem a execução com uma mensagem clara se algo estiver errado (download corrompido, arquivo trocado), em vez de deixar o erro aparecer lá na frente, de forma confusa.

In [3]:
df = pd.read_csv(ARQUIVO_BRUTO)

assert df.shape == (1_000_000, 32), f"Formato inesperado: {df.shape}"
assert set(df["month"].unique()) == set(range(8)), "Esperava os meses 0 a 7"
assert df["fraud_bool"].isin([0, 1]).all(), "O alvo deveria ser binário (0/1)"

print(f"OK: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

OK: 1,000,000 linhas x 32 colunas


## 3. Separação temporal

O modelo vai ser usado para prever pedidos **futuros**, então a avaliação precisa imitar isso: treinamos no passado e testamos no futuro. Seguimos o protocolo do artigo da base — **meses 0–5 para treino, 6–7 para teste**.

A partir daqui, `test.parquet` só é aberto **uma vez**, na avaliação final do modelo. Por isso imprimimos apenas os tamanhos, sem nenhuma estatística do teste.

In [4]:
MESES_TREINO = range(0, 6)  # meses 0–5
MESES_TESTE = range(6, 8)   # meses 6–7

treino = df[df["month"].isin(MESES_TREINO)].reset_index(drop=True)
teste = df[df["month"].isin(MESES_TESTE)].reset_index(drop=True)
assert len(treino) + len(teste) == len(df)

treino.to_parquet(DATA_DIR / "train.parquet", index=False)
teste.to_parquet(DATA_DIR / "test.parquet", index=False)

print(f"Treino: {len(treino):,} linhas ({len(treino) / len(df):.1%}) -> {DATA_DIR / 'train.parquet'}")
print(f"Teste:  {len(teste):,} linhas ({len(teste) / len(df):.1%}) -> {DATA_DIR / 'test.parquet'}")

Treino: 794,989 linhas (79.5%) -> c:\Users\joaog\Documents\Projetos\fraude-abertura-conta\data\train.parquet
Teste:  205,011 linhas (20.5%) -> c:\Users\joaog\Documents\Projetos\fraude-abertura-conta\data\test.parquet
